# Lab 2: Parallel Edges (Applied Version)
In this notebook, evolving  **Smart Content Generation System** by introducing parallel review steps.

Instead of just counting words, we want to analyze the draft concurrently for two distinct qualities:
1. **SEO Optimization** (identifying search-friendly structure).
2. **Readability & Tone** (checking how easy it is to read).

By using parallel execution, we evaluate both qualities simultaneously, shortening execution time.

We use the following concepts:
- **State Reducers**: `operator.add` to accumulate analysis logs from parallel execution paths.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### 1. Define State and Nodes
Notice  use `Annotated[List[str], operator.add]` for the `analysis_logs` field so parallel updates append to the list rather than overwriting.

In [2]:
from typing import TypedDict, List, Annotated
import operator
from mock_llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

class ContentState(TypedDict):
    topic: str
    draft: str
    analysis_logs: Annotated[List[str], operator.add]
    status: str

llm = get_llm(model="gpt-4o-mini", temperature=0.5)

def generate_draft_node(state: ContentState):
    print("--- Node: Generating Draft ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Write a brief, engaging paragraph (under 100 words) about the topic."),
        ("human", "Topic: {topic}")
    ])
    response = (prompt | llm).invoke({"topic": state["topic"]})
    return {"draft": response.content.strip(), "status": "drafted"}

# Parallel Node A: SEO Audit
def seo_audit_node(state: ContentState):
    print("--- Node: Conducting SEO Audit ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Evaluate this text for SEO optimization. List 1 strength and 1 improvement recommendation. Keep it under 40 words."),
        ("human", "{draft}")
    ])
    response = (prompt | llm).invoke({"draft": state["draft"]})
    return {"analysis_logs": [f"[SEO Audit]: {response.content.strip()}"]}

# Parallel Node B: Readability Check
def readability_check_node(state: ContentState):
    print("--- Node: Checking Readability ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Check the readability score/grade level and sentence flow. Provide a one-sentence critique. Keep it under 40 words."),
        ("human", "{draft}")
    ])
    response = (prompt | llm).invoke({"draft": state["draft"]})
    return {"analysis_logs": [f"[Readability Check]: {response.content.strip()}"]}

# Join Node: Combine Results
def finalize_analysis_node(state: ContentState):
    print("--- Node: Finalizing Analyses ---")
    return {"status": "reviewed"}

--- OpenAI API connection failed (Error code: 429 - {'error': {'message': 'You exceeded your c...). Falling back to Mock LLM ---


### 2. Build Graph with Parallel Branches
Let's wire `generate_draft` to point to BOTH `seo_audit` and `readability_check` in parallel, then merge them back into `finalize_analysis`.

In [3]:
builder = StateGraph(ContentState)
builder.add_node("generate_draft", generate_draft_node)
builder.add_node("seo_audit", seo_audit_node)
builder.add_node("readability_check", readability_check_node)
builder.add_node("finalize", finalize_analysis_node)

builder.add_edge(START, "generate_draft")

# Branch out to parallel nodes
builder.add_edge("generate_draft", "seo_audit")
builder.add_edge("generate_draft", "readability_check")

# Merge parallel branches into join node
builder.add_edge("seo_audit", "finalize")
builder.add_edge("readability_check", "finalize")

builder.add_edge("finalize", END)

graph = builder.compile()

try:
    graph.get_graph().print_ascii()
except:
    pass

                  +-----------+                
                  | __start__ |                
                  +-----------+                
                        *                      
                        *                      
                        *                      
                +----------------+             
                | generate_draft |             
                +----------------+             
                 **            **              
               **                **            
             **                    **          
+-------------------+           +-----------+  
| readability_check |           | seo_audit |  
+-------------------+           +-----------+  
                 **            **              
                   **        **                
                     **    **                  
                  +----------+                 
                  | finalize |                 
                  +----------+          

### 3. Run Parallel Workflow

In [4]:
result = graph.invoke({"topic": "The future of Web3 and decentralization", "draft": "", "analysis_logs": [], "status": "pending"})

print("\n--- Draft Generated ---")
print(result["draft"])

print("\n--- Analysis Logs (Combined via Reducer) ---")
for log in result["analysis_logs"]:
    print(log)

--- Node: Generating Draft ---
--- Node: Checking Readability ---
--- Node: Conducting SEO Audit ---
--- Node: Finalizing Analyses ---

--- Draft Generated ---
The future of Web3 lies in full decentralization. By moving authority away from singular tech platforms, web applications become trustless, secure, and user-owned. Through smart contracts, Web3 introduces transparent finance and community governance.

--- Analysis Logs (Combined via Reducer) ---
[Readability Check]: The future of Web3 lies in full decentralization. By moving authority away from singular tech platforms, web applications become trustless, secure, and user-owned. Through smart contracts, Web3 introduces transparent finance and community governance.
[SEO Audit]: The future of Web3 lies in full decentralization. By moving authority away from singular tech platforms, web applications become trustless, secure, and user-owned. Through smart contracts, Web3 introduces transparent finance and community governance.
